In [2]:
!pip install langchain-text-splitters langchain-experimental


In [3]:
!pip install langchain langgraph langchain-groq selenium beautifulsoup4 langchain-community langchain-huggingface chromadb sentence-transformers

  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached selenium-4.43.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached chromadb-1.5.8-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached build-1.4.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.2 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.41.0-py3-none-any.whl.metadata (2.6 kB)
  Usi

In [4]:
!pip install -q langchain-experimental

In [1]:
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "api_key"

In [6]:
import sys
!apt-get update
!apt-get install -y wget unzip
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get -fy install
!apt-get install -y chromium-chromedriver

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,977 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:14 https://ppa.launch

In [7]:
from pathlib import Path
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from selenium import webdriver
from bs4 import BeautifulSoup

In [8]:
def scrape_website(url):
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)
    html = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [9]:
# scrape_website("https://beautiful-soup-4.readthedocs.io/en/latest/#quick-start")

In [10]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)
def explain_content(text):
    prompt = f"""
    Explain the following website content in simple terms:

    {text}
    """
    return llm.invoke(prompt).content

In [11]:
!pip install sentence-transformers

In [12]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [13]:
def rerank(query, docs_with_scores, top_k=5):
    docs = [doc for doc, _ in docs_with_scores]

    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    reranked = list(zip(docs, scores))
    reranked.sort(key=lambda x: x[1], reverse=True)

    return reranked[:top_k]

In [14]:
VECTOR_DB_DIR = (Path.cwd() / "vector_db")
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(
    collection_name="rag_docs",
    embedding_function=embedding_model,
    persist_directory=str(VECTOR_DB_DIR),
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Loaded RAG vector store from {VECTOR_DB_DIR}")

@tool
def scrape(url):
    """Scrape a webpage and return its content."""
    return scrape_website(url)

@tool
def explain(text):
    """Explain given text content."""
    return explain_content(text)

RAG_SCORE_THRESHOLD = 0.8

@tool
def rag(query: str, url_filter: str | None = None):
    """Answer a question using the local RAG vector store with reranking."""
    search_kwargs = {"k": 20}
    if url_filter:
        search_kwargs["filter"] = {"url": url_filter}

    matches = vectorstore.similarity_search_with_score(query, **search_kwargs)
    if not matches:
        return "I don't know based on the indexed data."

    relevant_matches = [
        (doc, score) for doc, score in matches if score <= RAG_SCORE_THRESHOLD
    ]
    if not relevant_matches:
        return "I don't know based on the indexed data."
    reranked = rerank(query, relevant_matches, top_k=5)


    context_lines = []
    for rank, (doc, score) in enumerate(relevant_matches, start=1):
        source_url = doc.metadata.get("url", "unknown")
        context_lines.append(f"Source {rank} | url: {source_url} | score: {score:.4f}\n{doc.page_content}")

    context_text = "\n\n".join(context_lines)
    prompt = f"""
    You are a strict RAG assistant.

    Answer ONLY using the context below.
    If the answer is not clearly present, say:
    "I don't know based on the indexed data."

    Question:
    {query}

    Context:
    {context_text}

    Answer:
    """
    return llm.invoke(prompt).content

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1883/148522039.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Loaded RAG vector store from /content/vector_db


In [15]:
class AgentState(TypedDict):
    tools: str
    inputs: str
    last_task:str
    web_url: str
    scraped_content: str
    last_response: str
    task: str
    history: list
    memory_window: int

In [16]:
TOOLS = """
You have access to the following tools:

1. scrape(url: str)
   - Use this to extract content from a website URL.

2. explain(text: str)
   - Use this to explain or summarize given content.

3. rag(query: str)
   - Use this for factual questions about indexed pages or stored knowledge.
   - Prefer this over explain when the user is asking a question about a scraped page.
   - It should answer only from retrieved chunks in the vector database.


Decide which tool to use based on the user input.
If no tool is needed, return: finish
"""

In [29]:
from langchain_experimental.text_splitter import SemanticChunker

window_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)
semantic_splitter = SemanticChunker(embedding_model)

def scrape_node(state: AgentState):
  content = scrape.invoke({"url": state["web_url"]})
  window_chunks = window_splitter.split_text(content)

  docs = []
  for chunk in window_chunks:
      sem_docs = semantic_splitter.create_documents([chunk])
      docs.extend(sem_docs)



  for doc in docs:
      doc.metadata = {"url": state["web_url"], "source": "web_scrape"}

  vectorstore.delete(where={"url": state["web_url"]})
  vectorstore.add_documents(docs)

  return {
      "scraped_content": content,
      "last_response": f"Sliding window + semantic chunking → {len(docs)} chunks."
  }

In [20]:
def explain_node(state: AgentState):
    explanation = explain.invoke({"text": state["scraped_content"]})
    print(explanation)
    return {
        "last_response": explanation
    }

In [21]:
def rag_node(state: AgentState):
    answer = rag.invoke({"query": state["inputs"], "url_filter": state.get("web_url")})
    print(answer)
    return {
        "last_response": answer
    }

In [22]:
def rule_based_task(state: AgentState) -> str:
    user_text = state["inputs"].strip().lower()
    has_scraped_content = bool(state.get("scraped_content"))
    has_url_context = bool(state.get("web_url"))
    last_task = state.get("last_task", "")

    asks_history = any(phrase in user_text for phrase in ["previous", "earlier", "last response", "what did you say", "from history", "before"])
    asks_scrape = any(phrase in user_text for phrase in ["scrape", "fetch", "load", "index", "crawl"])
    asks_explain = any(phrase in user_text for phrase in ["explain", "summarize", "summary", "describe"])

    question_starts = ("what", "why", "how", "when", "where", "who", "which", "compare", "difference")
    is_question = "?" in user_text or user_text.startswith(question_starts)

    if asks_history:
        return "recall"
    if asks_explain and last_task == "scrape" and has_scraped_content:
        return "explain"
    if asks_scrape and last_task != "scrape":
        return "scrape"
    if is_question and (has_scraped_content or has_url_context):
        return "rag"
    if asks_explain and has_scraped_content:
        return "explain"
    return ""

def brain(state: AgentState):
    rule_choice = rule_based_task(state)
    if rule_choice:
        return {"task": rule_choice, "last_task": rule_choice}

    formatted_history = "\n".join([f"User: {t['user']} | Agent: {t['response']}" for t in state['history']])
    prompt = f"""
    You are an agent. Decisions: scrape, explain, rag, recall, finish.
    History: {formatted_history}
    Input: {state['inputs']}
    Last Task: {state['last_task']}
    Return one word only.
    """
    raw_decision = llm.invoke(prompt).content.strip().lower()
    decision = raw_decision.split()[0] if raw_decision else "finish"

    if decision not in {"scrape", "explain", "rag", "recall", "finish"}:
        decision = "finish"

    return {"task": decision, "last_task": decision}

In [23]:
def recall_node(state: AgentState):
    formatted_history = []
    for turn in state['history']:
        formatted_history.append(
            f"User: {turn.get('user', '')} | Task: {turn.get('task', '')} | Agent: {turn.get('response', '')}"
        )
    formatted_history_text = "\n".join(formatted_history)

    prompt = f"""
You answer questions using the conversation memory below.

Conversation memory:
{formatted_history_text}

Current user question:
{state['inputs']}

Answer only from the stored memory. If the answer is not present, say you do not have that earlier response in memory.
"""

    answer = llm.invoke(prompt).content.strip()
    print(answer)
    return {
        "last_response": answer
    }

In [24]:
def route_task(state: AgentState):
    print(state["task"], " called \n")
    return state["task"]

In [25]:
builder = StateGraph(AgentState)

builder.add_node("brain", brain)
builder.add_node("scrape", scrape_node)
builder.add_node("explain", explain_node)
builder.add_node("rag", rag_node)
builder.add_node("recall", recall_node)

builder.set_entry_point("brain")

builder.add_conditional_edges(
    "brain",
    route_task,
    {
        "scrape": "scrape",
        "explain": "explain",
        "rag": "rag",
        "recall": "recall",
        "finish": END,
    },
)

builder.add_edge("scrape", "brain")
builder.add_edge("explain", END)
builder.add_edge("rag", END)
builder.add_edge("recall", END)

graph = builder.compile()

In [26]:
state = {
    "inputs": "",
    "tools": TOOLS,
    "web_url": "https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant",
    "scraped_content": "",
    "last_response": "",
    "task": "",
    "history": [],
    "last_task": "",
    "memory_window": 5
}

def run_agent(user_text, web_url = None):
    global state

    state["inputs"] = user_text
    if web_url is not None:
        state["web_url"] = web_url

    result = graph.invoke(state)
    state.update(result)

    executed_task = state.get("task", "finish")
    state["last_task"] = executed_task
    state["history"].append({
        "user": user_text,
        "task": executed_task,
        "response": state.get("last_response", ""),
    })

    window = state.get("memory_window", 5)
    state["history"] = state["history"][-window:]

    return state

In [ ]:
run_agent("scrape this website ","https://en.wikipedia.org/wiki/Main_Page" )

scrape  called 

explain  called 

Here's a simple explanation of the content on the Wikipedia homepage:

---

### **What is Wikipedia?**
- **Free Encyclopedia**: Wikipedia is a free, online encyclopedia where anyone can read and edit articles. It’s run by volunteers and hosted by a nonprofit organization called the **Wikimedia Foundation**.
- **Big Community**: Over **275,000 active editors** help create and update the **7.1 million English articles** (and even more in other languages).

---

### **Featured Article: Nihilism**
- **What is Nihilism?**  
  Nihilism is a philosophy that questions the meaning, purpose, or truth of life. The article explains different types:
  - **Existential Nihilism**: Life has no inherent meaning.
  - **Moral Nihilism**: Morality doesn’t exist objectively.
  - **Epistemological Nihilism**: Truth and knowledge are uncertain.
  - **Cosmological Nihilism**: The universe is indifferent to humans.
  - **Metaphysical Nihilism**: There’s no reason why anything

{'inputs': 'scrape this website ',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this for factual questions about indexed pages or stored knowledge.\n   - Prefer this over explain when the user is asking a question about a scraped page.\n   - It should answer only from retrieved chunks in the vector database.\n\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Main_Page',
 'scraped_content': 'Wikipedia, the free encyclopedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Search Search Appearance Appearance move to sidebar hide

In [28]:
run_agent("explain this website")

explain  called 

Here's a simple explanation of the content on the Wikipedia homepage:

---

### **What is Wikipedia?**
Wikipedia is a **free online encyclopedia** where anyone can edit or add information. It has **7.1 million English articles** and is run by volunteers. You can read about almost any topic, from history to science to pop culture.

---

### **Key Features on the Page**
1. **Navigation Menu**  
   - Links to the **Main Page**, **Current Events**, **Random Article**, and other tools.  
   - Options to **create an account**, **log in**, or **donate** to support Wikipedia.

2. **Search Bar**  
   - You can search for any topic.  
   - Customize how the page looks (font size, color theme, etc.).

3. **Featured Article**  
   - **Today’s Topic**: *Nihilism* (a philosophy that questions meaning, morality, and knowledge).  
     - **Existential nihilism**: Life has no inherent purpose.  
     - **Moral nihilism**: Morality is not real.  
     - **Cosmological nihilism**: The u

{'inputs': 'explain this website',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this for factual questions about indexed pages or stored knowledge.\n   - Prefer this over explain when the user is asking a question about a scraped page.\n   - It should answer only from retrieved chunks in the vector database.\n\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Main_Page',
 'scraped_content': 'Wikipedia, the free encyclopedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Search Search Appearance Appearance move to sidebar hide

In [40]:
run_agent("what is Wikipedia?")

rag  called 

Wikipedia is a free encyclopedia that anyone can edit, hosted by the Wikimedia Foundation and written by volunteer editors. It is described as "the free encyclopedia that anyone can edit" (Source 7) and is part of a network of Wikimedia projects, including Wikibooks, Wikidata, and others (Source 1).


{'inputs': 'what is Wikipedia?',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this for factual questions about indexed pages or stored knowledge.\n   - Prefer this over explain when the user is asking a question about a scraped page.\n   - It should answer only from retrieved chunks in the vector database.\n\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Main_Page',
 'scraped_content': 'Wikipedia, the free encyclopedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Search Search Appearance Appearance move to sidebar hide T